# ML/AI Suitability Modeling — West Java Agro-Industrial Residue Potential

Revised notebook for machine learning baseline modeling of grid-level suitability scores and potential classes.

Main revision:
- Does not use `mean_squared_error(..., squared=False)` because some `scikit-learn` versions do not support the `squared` argument.
- RMSE is computed using a compatible function: `root_mean_squared_error` when available, or `sqrt(mean_squared_error)` otherwise.
- Adds checks for Excel sheets, features, targets, missing values, and prediction exports.

Main input:
- `grid_suitability_jawa_barat_final.xlsx`
- Optional: `grid_suitability_jawa_barat_final.gpkg` for exporting predictions to GeoPackage.


In [ ]:
# ============================================================
# 1. Install required libraries
# ============================================================

!pip install pandas numpy scikit-learn openpyxl matplotlib joblib geopandas pyogrio -q

# XGBoost is optional. If installation or import fails, the notebook continues with Random Forest and HistGradientBoosting models.
try:
    import xgboost
    print('xgboost already installed:', xgboost.__version__)
except Exception:
    !pip install xgboost -q


In [ ]:
# ============================================================
# 2. Import libraries
# ============================================================

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from google.colab import drive, files

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier,
    HistGradientBoostingRegressor,
    HistGradientBoostingClassifier
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings('ignore')

try:
    from sklearn.metrics import root_mean_squared_error
    HAS_ROOT_RMSE = True
except Exception:
    HAS_ROOT_RMSE = False

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

print('HAS_ROOT_RMSE:', HAS_ROOT_RMSE)
print('HAS_XGBOOST:', HAS_XGBOOST)


In [ ]:
# ============================================================
# 3. Mount Google Drive and set paths
# ============================================================

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/geospatial_biomass_carbon'
OUTPUT_DIR = os.path.join(BASE_DIR, 'ml_outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXCEL_FILE = os.path.join(BASE_DIR, 'grid_suitability_jawa_barat_final.xlsx')
GPKG_FILE = os.path.join(BASE_DIR, 'grid_suitability_jawa_barat_final.gpkg')  # optional

OUTPUT_PRED_XLSX = os.path.join(OUTPUT_DIR, 'grid_suitability_jawa_barat_ml_prediction.xlsx')
OUTPUT_PRED_CSV = os.path.join(OUTPUT_DIR, 'grid_suitability_jawa_barat_ml_prediction.csv')
OUTPUT_AUDIT_XLSX = os.path.join(OUTPUT_DIR, 'dataset_audit_summary.xlsx')
OUTPUT_MODEL_REG = os.path.join(OUTPUT_DIR, 'best_regression_model.joblib')
OUTPUT_MODEL_CLF = os.path.join(OUTPUT_DIR, 'best_classification_model.joblib')
OUTPUT_GPKG = os.path.join(OUTPUT_DIR, 'grid_suitability_jawa_barat_ml_prediction.gpkg')

if not os.path.exists(EXCEL_FILE):
    raise FileNotFoundError(f'File not found: {EXCEL_FILE}')

print('Input Excel:', EXCEL_FILE)
print('Output folder:', OUTPUT_DIR)


In [ ]:
# ============================================================
# 4. Load final dataset
# ============================================================

xl = pd.ExcelFile(EXCEL_FILE)
print('Sheets:', xl.sheet_names)

# Use the `Grid_Final` sheet when available. Otherwise, use the first non-README sheet.
if 'Grid_Final' in xl.sheet_names:
    SHEET_NAME = 'Grid_Final'
else:
    candidates = [s for s in xl.sheet_names if 'readme' not in s.lower()]
    SHEET_NAME = candidates[0] if candidates else xl.sheet_names[0]

df = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME)
print('Using sheet:', SHEET_NAME)
print('Dataset shape:', df.shape)
display(df.head())
print(df.columns.tolist())


In [ ]:
# ============================================================
# 5. Audit final dataset
# ============================================================

required_targets = ['suitability_score_final', 'potential_class_final']
missing_targets = [c for c in required_targets if c not in df.columns]
if missing_targets:
    raise ValueError(f'Target columns not found: {missing_targets}')

audit_rows = []

def add_audit(item, value):
    audit_rows.append({'item': item, 'value': value})

add_audit('rows', len(df))
add_audit('columns', len(df.columns))
add_audit('unique_grid_id', df['grid_id'].nunique() if 'grid_id' in df.columns else np.nan)
add_audit('duplicate_grid_id', df['grid_id'].duplicated().sum() if 'grid_id' in df.columns else np.nan)
add_audit('district_count', df['district'].nunique() if 'district' in df.columns else np.nan)
add_audit('missing_suitability_score_final', df['suitability_score_final'].isna().sum())
add_audit('missing_potential_class_final', df['potential_class_final'].isna().sum())

audit_df = pd.DataFrame(audit_rows)
class_dist = df['potential_class_final'].value_counts(dropna=False).reset_index()
class_dist.columns = ['potential_class_final', 'count']

missing_summary = df.isna().sum().reset_index()
missing_summary.columns = ['column', 'missing_count']
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_count', ascending=False)

display(audit_df)
display(class_dist)
display(missing_summary.head(30))


In [ ]:
# ============================================================
# 6. Define targets and features
# ============================================================

REG_TARGET = 'suitability_score_final'
CLF_TARGET = 'potential_class_final'
GROUP_COL = 'district'
ID_COL = 'grid_id'

# Input features are kept relatively raw and explanatory.
# Directly derived features from the suitability formula, such as residue_score and road_accessibility_score,
# agroindustry_proximity_score, cropland_score, ntl_score are intentionally not used as main features
# so the model does not simply reproduce the rule-based formula too directly.

candidate_features = [
    # Remote sensing features
    'mean_NDVI', 'mean_EVI', 'mean_NDWI', 'mean_NBR', 'mean_SAVI',
    'mean_VIIRS_NTL', 'cropland_ratio', 'landcover_majority', 'landcover_name',
    'environmental_score',

    # Accessibility and infrastructure
    'distance_to_road_km', 'distance_to_agroindustry_km',
    'agroindustry_count_5km', 'agroindustry_count_10km',
    'nearest_road_fclass',

    # Residue and agroindustry context
    'estimated_grid_residue_ton_year', 'total_residue_ton_year',
    'dominant_commodity', 'dominant_residue',
    'nearest_agroindustry_type', 'nearest_agroindustry_commodity',

    # Spatial/grid context
    'coverage_ratio', 'area_km2', 'centroid_lon', 'centroid_lat'
]

feature_cols = [c for c in candidate_features if c in df.columns]
missing_features = [c for c in candidate_features if c not in df.columns]

print('Selected feature count:', len(feature_cols))
print('Selected features:', feature_cols)
print('Missing candidate features:', missing_features)

if len(feature_cols) == 0:
    raise ValueError('No input features were found.')

model_df = df.copy()
model_df = model_df.dropna(subset=[REG_TARGET, CLF_TARGET]).copy()

# Separate numeric and categorical features
numeric_features = [c for c in feature_cols if pd.api.types.is_numeric_dtype(model_df[c])]
categorical_features = [c for c in feature_cols if c not in numeric_features]

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)


In [ ]:
# ============================================================
# 7. Data split: district-based spatial split
# ============================================================

X = model_df[feature_cols].copy()
y_reg = model_df[REG_TARGET].copy()
y_clf_raw = model_df[CLF_TARGET].copy()
groups = model_df[GROUP_COL].astype(str).copy() if GROUP_COL in model_df.columns else pd.Series(np.arange(len(model_df)))

# Encode label kelas
label_order = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']
label_encoder = LabelEncoder()
label_encoder.fit([c for c in label_order if c in y_clf_raw.unique()] + [c for c in y_clf_raw.unique() if c not in label_order])
y_clf = label_encoder.transform(y_clf_raw.astype(str))

# Spatial split: train 70%, temp 30% berdasarkan district
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(X, y_reg, groups=groups))

X_train = X.iloc[train_idx].copy()
y_reg_train = y_reg.iloc[train_idx].copy()
y_clf_train = y_clf[train_idx]
groups_temp = groups.iloc[temp_idx].copy()

X_temp = X.iloc[temp_idx].copy()
y_reg_temp = y_reg.iloc[temp_idx].copy()
y_clf_temp = y_clf[temp_idx]

# Split temp menjadi validation dan test masing-masing setengah
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=43)
val_rel_idx, test_rel_idx = next(gss2.split(X_temp, y_reg_temp, groups=groups_temp))

X_val = X_temp.iloc[val_rel_idx].copy()
y_reg_val = y_reg_temp.iloc[val_rel_idx].copy()
y_clf_val = y_clf_temp[val_rel_idx]

X_test = X_temp.iloc[test_rel_idx].copy()
y_reg_test = y_reg_temp.iloc[test_rel_idx].copy()
y_clf_test = y_clf_temp[test_rel_idx]

print('Train:', X_train.shape, 'Districts:', groups.iloc[train_idx].nunique())
print('Val  :', X_val.shape, 'Districts:', groups_temp.iloc[val_rel_idx].nunique())
print('Test :', X_test.shape, 'Districts:', groups_temp.iloc[test_rel_idx].nunique())
print('Train districts:', sorted(groups.iloc[train_idx].unique()))
print('Val districts  :', sorted(groups_temp.iloc[val_rel_idx].unique()))
print('Test districts :', sorted(groups_temp.iloc[test_rel_idx].unique()))


In [ ]:
# ============================================================
# 8. Preprocessing Pipeline
# ============================================================

# The `OneHotEncoder` API differs across scikit-learn versions.
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', ohe)
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

print('Preprocessor ready.')


In [ ]:
# ============================================================
# 9. Regression models for suitability score
# ============================================================

# Important revision:
# Do not use `mean_squared_error(..., squared=False)` because some scikit-learn versions
# the `squared` argument is not available. The function below is compatible across versions.

def safe_rmse(y_true, y_pred):
    if HAS_ROOT_RMSE:
        return root_mean_squared_error(y_true, y_pred)
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def regression_metrics(y_true, y_pred):
    return {
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'RMSE': float(safe_rmse(y_true, y_pred)),
        'R2': float(r2_score(y_true, y_pred))
    }

regression_models = {
    'RandomForestRegressor': RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        max_depth=None,
        min_samples_leaf=2
    ),
    'HistGradientBoostingRegressor': HistGradientBoostingRegressor(
        random_state=42,
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31
    )
}

if HAS_XGBOOST:
    regression_models['XGBRegressor'] = XGBRegressor(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )

regression_results = []
regression_pipelines = {}

for name, model in regression_models.items():
    print('Training:', name)
    pipe = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_reg_train)

    pred_val = pipe.predict(X_val)
    pred_test = pipe.predict(X_test)

    val_metrics = regression_metrics(y_reg_val, pred_val)
    test_metrics = regression_metrics(y_reg_test, pred_test)

    regression_results.append({
        'model': name,
        'split': 'validation',
        **val_metrics
    })
    regression_results.append({
        'model': name,
        'split': 'test',
        **test_metrics
    })

    regression_pipelines[name] = pipe

regression_results_df = pd.DataFrame(regression_results)
display(regression_results_df)

# Select model terbaik berdasarkan RMSE validation terkecil
val_reg = regression_results_df[regression_results_df['split'] == 'validation'].copy()
best_reg_name = val_reg.sort_values('RMSE', ascending=True).iloc[0]['model']
best_reg_model = regression_pipelines[best_reg_name]

print('Best regression model:', best_reg_name)


In [ ]:
# ============================================================
# 10. Classification models for potential class
# ============================================================

def classification_metrics(y_true, y_pred):
    return {
        'Accuracy': float(accuracy_score(y_true, y_pred)),
        'Macro_F1': float(f1_score(y_true, y_pred, average='macro')),
        'Weighted_F1': float(f1_score(y_true, y_pred, average='weighted')),
        'Macro_Precision': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'Macro_Recall': float(recall_score(y_true, y_pred, average='macro', zero_division=0))
    }

classification_models = {
    'RandomForestClassifier': RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        max_depth=None,
        min_samples_leaf=2,
        class_weight='balanced'
    ),
    'HistGradientBoostingClassifier': HistGradientBoostingClassifier(
        random_state=42,
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31
    )
}

if HAS_XGBOOST:
    classification_models['XGBClassifier'] = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    )

classification_results = []
classification_pipelines = {}

for name, model in classification_models.items():
    print('Training:', name)
    pipe = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_clf_train)

    pred_val = pipe.predict(X_val)
    pred_test = pipe.predict(X_test)

    val_metrics = classification_metrics(y_clf_val, pred_val)
    test_metrics = classification_metrics(y_clf_test, pred_test)

    classification_results.append({
        'model': name,
        'split': 'validation',
        **val_metrics
    })
    classification_results.append({
        'model': name,
        'split': 'test',
        **test_metrics
    })

    classification_pipelines[name] = pipe

classification_results_df = pd.DataFrame(classification_results)
display(classification_results_df)

# Select model terbaik berdasarkan Macro_F1 validation tertinggi
val_clf = classification_results_df[classification_results_df['split'] == 'validation'].copy()
best_clf_name = val_clf.sort_values('Macro_F1', ascending=False).iloc[0]['model']
best_clf_model = classification_pipelines[best_clf_name]

print('Best classification model:', best_clf_name)


In [ ]:
# ============================================================
# 11. Evaluasi Detail Model Terbaik
# ============================================================

# Regression detail
pred_reg_test = best_reg_model.predict(X_test)
print('Best Regression Model:', best_reg_name)
print(regression_metrics(y_reg_test, pred_reg_test))

plt.figure(figsize=(6, 6))
plt.scatter(y_reg_test, pred_reg_test, s=5, alpha=0.5)
plt.xlabel('Observed suitability_score_final')
plt.ylabel('Predicted suitability_score_final')
plt.title(f'Regression Test: {best_reg_name}')
plt.grid(True, alpha=0.3)
plt.tight_layout()
reg_scatter_path = os.path.join(OUTPUT_DIR, 'regression_observed_vs_predicted_test.png')
plt.savefig(reg_scatter_path, dpi=300)
plt.show()

# Classification detail
pred_clf_test = best_clf_model.predict(X_test)
print('Best Classification Model:', best_clf_name)
print(classification_report(y_clf_test, pred_clf_test, target_names=label_encoder.classes_, zero_division=0))

cm = confusion_matrix(y_clf_test, pred_clf_test)
plt.figure(figsize=(7, 6))
plt.imshow(cm)
plt.title(f'Confusion Matrix: {best_clf_name}')
plt.xlabel('Predicted')
plt.ylabel('Observed')
plt.xticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45, ha='right')
plt.yticks(range(len(label_encoder.classes_)), label_encoder.classes_)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix_test.png')
plt.savefig(cm_path, dpi=300)
plt.show()


In [ ]:
# ============================================================
# 12. Feature Importance
# ============================================================

def get_feature_names_from_preprocessor(preprocessor, numeric_features, categorical_features):
    out_features = []
    out_features.extend(numeric_features)

    if len(categorical_features) > 0:
        try:
            ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
            cat_names = list(ohe.get_feature_names_out(categorical_features))
            out_features.extend(cat_names)
        except Exception:
            out_features.extend(categorical_features)

    return out_features

def plot_feature_importance(pipe, title, output_path, top_n=25):
    model = pipe.named_steps['model']
    preprocess = pipe.named_steps['preprocess']

    if not hasattr(model, 'feature_importances_'):
        print(f'Model {title} does not provide `feature_importances_`.')
        return pd.DataFrame()

    names = get_feature_names_from_preprocessor(preprocess, numeric_features, categorical_features)
    importances = model.feature_importances_

    if len(names) != len(importances):
        names = [f'feature_{i}' for i in range(len(importances))]

    imp_df = pd.DataFrame({
        'feature': names,
        'importance': importances
    }).sort_values('importance', ascending=False)

    top = imp_df.head(top_n).iloc[::-1]
    plt.figure(figsize=(8, 8))
    plt.barh(top['feature'], top['importance'])
    plt.title(title)
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.show()

    return imp_df

reg_imp_path = os.path.join(OUTPUT_DIR, 'feature_importance_regression.png')
clf_imp_path = os.path.join(OUTPUT_DIR, 'feature_importance_classification.png')

reg_importance_df = plot_feature_importance(best_reg_model, f'Feature Importance Regression: {best_reg_name}', reg_imp_path)
clf_importance_df = plot_feature_importance(best_clf_model, f'Feature Importance Classification: {best_clf_name}', clf_imp_path)

display(reg_importance_df.head(20) if len(reg_importance_df) else reg_importance_df)
display(clf_importance_df.head(20) if len(clf_importance_df) else clf_importance_df)


In [ ]:
# ============================================================
# 13. Predict the complete grid and export results
# ============================================================

X_all = model_df[feature_cols].copy()

model_df['ml_regression_model'] = best_reg_name
model_df['ml_predicted_suitability_score'] = best_reg_model.predict(X_all)
model_df['ml_prediction_error_rule_based'] = model_df['ml_predicted_suitability_score'] - model_df[REG_TARGET]

model_df['ml_classification_model'] = best_clf_name
ml_class_pred = best_clf_model.predict(X_all)
model_df['ml_predicted_potential_class'] = label_encoder.inverse_transform(ml_class_pred)
model_df['ml_class_match_rule_based'] = model_df['ml_predicted_potential_class'].astype(str) == model_df[CLF_TARGET].astype(str)

# Create classes from regression predictions using quantiles for mapping
try:
    model_df['ml_regression_potential_class'] = pd.qcut(
        model_df['ml_predicted_suitability_score'],
        q=5,
        labels=['Very Low', 'Low', 'Moderate', 'High', 'Very High'],
        duplicates='drop'
    ).astype(str)
except Exception:
    model_df['ml_regression_potential_class'] = pd.cut(
        model_df['ml_predicted_suitability_score'],
        bins=[-np.inf, 0.2, 0.4, 0.6, 0.8, np.inf],
        labels=['Very Low', 'Low', 'Moderate', 'High', 'Very High']
    ).astype(str)

# Save CSV and Excel outputs
model_df.to_csv(OUTPUT_PRED_CSV, index=False)

with pd.ExcelWriter(OUTPUT_PRED_XLSX, engine='openpyxl') as writer:
    model_df.to_excel(writer, sheet_name='ML_Predictions', index=False)
    audit_df.to_excel(writer, sheet_name='Audit', index=False)
    class_dist.to_excel(writer, sheet_name='Class_Distribution', index=False)
    regression_results_df.to_excel(writer, sheet_name='Regression_Results', index=False)
    classification_results_df.to_excel(writer, sheet_name='Classification_Results', index=False)
    if len(reg_importance_df):
        reg_importance_df.to_excel(writer, sheet_name='Reg_Feature_Importance', index=False)
    if len(clf_importance_df):
        clf_importance_df.to_excel(writer, sheet_name='Clf_Feature_Importance', index=False)

with pd.ExcelWriter(OUTPUT_AUDIT_XLSX, engine='openpyxl') as writer:
    audit_df.to_excel(writer, sheet_name='Audit', index=False)
    missing_summary.to_excel(writer, sheet_name='Missing_Values', index=False)
    class_dist.to_excel(writer, sheet_name='Class_Distribution', index=False)
    regression_results_df.to_excel(writer, sheet_name='Regression_Results', index=False)
    classification_results_df.to_excel(writer, sheet_name='Classification_Results', index=False)

print('Saved:')
print(OUTPUT_PRED_XLSX)
print(OUTPUT_PRED_CSV)
print(OUTPUT_AUDIT_XLSX)

display(model_df[[ID_COL, GROUP_COL, REG_TARGET, 'ml_predicted_suitability_score', CLF_TARGET, 'ml_predicted_potential_class']].head())


In [ ]:
# ============================================================
# 14. Export GeoPackage for QGIS when available
# ============================================================

try:
    import geopandas as gpd

    if os.path.exists(GPKG_FILE):
        # Read the first layer from the GeoPackage
        layers = gpd.list_layers(GPKG_FILE)
        print('GPKG layers:')
        display(layers)
        layer_name = layers.iloc[0]['name'] if hasattr(layers, 'iloc') else layers[0]
        gdf = gpd.read_file(GPKG_FILE, layer=layer_name)
        print('GPKG loaded:', gdf.shape)

        # Merge prediksi berdasarkan grid_id
        pred_cols = [
            ID_COL,
            'ml_regression_model',
            'ml_predicted_suitability_score',
            'ml_prediction_error_rule_based',
            'ml_regression_potential_class',
            'ml_classification_model',
            'ml_predicted_potential_class',
            'ml_class_match_rule_based'
        ]

        gdf_out = gdf.merge(model_df[pred_cols], on=ID_COL, how='left')
        gdf_out.to_file(OUTPUT_GPKG, layer='grid_ml_prediction', driver='GPKG')
        print('GeoPackage saved:', OUTPUT_GPKG)
    else:
        print('GPKG input not found; GeoPackage export skipped:', GPKG_FILE)
except Exception as e:
    print('GeoPackage export failed:', e)


In [ ]:
# ============================================================
# 15. Save models and selected output files
# ============================================================

joblib.dump(best_reg_model, OUTPUT_MODEL_REG)
joblib.dump(best_clf_model, OUTPUT_MODEL_CLF)

print('Model saved:')
print(OUTPUT_MODEL_REG)
print(OUTPUT_MODEL_CLF)

print('
Output files in:', OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

# Download selected small output files. Large GeoPackage files can be retrieved manually from Google Drive.
files.download(OUTPUT_PRED_XLSX)
files.download(OUTPUT_AUDIT_XLSX)


## Interpretation notes

Main outputs to inspect after the notebook has finished:

1. `Regression_Results`: select the model with the lowest RMSE and highest R².
2. `Classification_Results`: select the model with the highest Macro-F1.
3. `Reg_Feature_Importance`: identifies the most influential factors for the suitability score.
4. `ML_Predictions`: is used for the machine-learning prediction map in QGIS.

Important methodological note:
- Target `suitability_score_final` dan `potential_class_final` adalah proxy-label hasil rule-based suitability index, bukan ground-truth lapangan.
- Model ML pada stage ini adalah baseline tabular geospatial ML.
- For deep learning models such as DeepLabV3+ or U-Net, the next stage is to build raster stacks and patch datasets.
